# **Day 82: NumPy Band Math - NDVI, NDWI and Image Statistics**

**Band math** means doing calculations with the bands of a satellite image. With NumPy, a formula is written **exactly as it appears on paper**, and it runs on **every pixel at once**.

Today we make a small **synthetic** (computer-made) satellite image of an area near the Sundarbans, with water, mangrove, farmland and a village, and analyse it.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

rows, cols = 60, 80
# A "true" land cover map: 1 = water, 2 = mangrove, 3 = agriculture, 4 = built-up
truth = np.full((rows, cols), 3)
truth[:, :18] = 1                         # A river on the left side
truth[:, 18:38] = 2                       # Mangrove along the river
truth[40:55, 55:72] = 4                   # A village
truth[5:15, 60:75] = 1                    # A pond

# Typical reflectance of each class: (green, red, nir, swir)
signature = {1: (0.08, 0.05, 0.02, 0.01), 2: (0.06, 0.03, 0.38, 0.14),
             3: (0.09, 0.08, 0.30, 0.22), 4: (0.12, 0.14, 0.20, 0.28)}

bands = {}
for i, name in enumerate(["green", "red", "nir", "swir"]):
    band = np.zeros((rows, cols))
    for cls, values in signature.items():
        band[truth == cls] = values[i]
    bands[name] = (band + rng.normal(0, 0.01, band.shape)).clip(0.001, 1)   # Add a little noise

green, red, nir, swir = bands["green"], bands["red"], bands["nir"], bands["swir"]
print("Each band shape:", red.shape)

Each band shape: (60, 80)


## **Broadcasting: Maths Between Different Shapes**
When we do maths with arrays of different shapes, NumPy **stretches** the smaller one to match. This is called **broadcasting**. The simplest case: a single number is applied to every pixel.

In [2]:
scaled = red * 10000              # One number and a whole band
print(scaled[0, :3].round())

gains = np.array([1.0, 1.1, 0.9, 1.05])          # One correction factor for each band
pixel = np.array([0.08, 0.05, 0.30, 0.20])       # One pixel's four band values
print(pixel * gains)

[579. 374. 563.]
[0.08  0.055 0.27  0.21 ]


## **NDVI: Normalized Difference Vegetation Index**
**NDVI = (NIR - Red) / (NIR + Red)**

Healthy plants absorb red light (for photosynthesis) and reflect NIR strongly. So:
- **High NDVI** (0.5 to 0.9): dense, healthy vegetation
- **Low NDVI** (0 to 0.2): soil, buildings
- **Negative NDVI**: water

In [3]:
ndvi = (nir - red) / (nir + red)
print("NDVI shape:", ndvi.shape)
print("NDVI range:", ndvi.min().round(2), "to", ndvi.max().round(2))
print("Mean NDVI:", ndvi.mean().round(3))

NDVI shape: (60, 80)
NDVI range: -0.97 to 0.99
Mean NDVI: 0.363


## **NDWI and NDBI**
- **NDWI** (water index) = (Green - NIR) / (Green + NIR). Water is **positive**.
- **NDBI** (built-up index) = (SWIR - NIR) / (SWIR + NIR). Buildings are **positive**.

In [4]:
ndwi = (green - nir) / (green + nir)
ndbi = (swir - nir) / (swir + nir)

for name, index in [("NDVI", ndvi), ("NDWI", ndwi), ("NDBI", ndbi)]:
    print(f"{name}: water {index[truth == 1].mean():+.2f}  mangrove {index[truth == 2].mean():+.2f}  "
          f"agriculture {index[truth == 3].mean():+.2f}  built-up {index[truth == 4].mean():+.2f}")

NDVI: water -0.45  mangrove +0.85  agriculture +0.58  built-up +0.18
NDWI: water +0.61  mangrove -0.73  agriculture -0.54  built-up -0.25
NDBI: water -0.31  mangrove -0.46  agriculture -0.15  built-up +0.17


Each index is good at finding one thing:
- NDVI is highest for **mangrove**.
- NDWI is highest (positive) only for **water**.
- NDBI is highest for **built-up** areas.

## **Classifying Pixels with np.where and np.select**
`np.where(condition, value_if_true, value_if_false)` works like `if-else` on every pixel.

In [5]:
water = np.where(ndwi > 0, 1, 0)
print("Water pixels:", water.sum())

# np.select: many conditions at once (like if-elif-elif-else)
conditions = [ndwi > 0, ndbi > 0, ndvi > 0.6]
choices = [1, 4, 2]                       # water, built-up, mangrove
classified = np.select(conditions, choices, default=3)     # Everything else: agriculture

Water pixels: 1230


## **How Good is Our Classification? (Accuracy)**
Because we made this image ourselves, we know the **true** class of every pixel. We can check what percentage of pixels we classified correctly.

In [6]:
correct = (classified == truth).mean()
print(f"Overall accuracy: {correct:.1%}")

names = {1: "Water", 2: "Mangrove", 3: "Agriculture", 4: "Built-up"}
pixel_area_ha = 10 * 10 / 10_000           # Sentinel-2 pixel (10 m) in hectare
for code, name in names.items():
    n = (classified == code).sum()
    print(f"{name:<12}{n:>6} pixels  {n * pixel_area_ha:>6.1f} ha")

Overall accuracy: 86.3%
Water         1230 pixels    12.3 ha
Mangrove      1856 pixels    18.6 ha
Agriculture   1459 pixels    14.6 ha
Built-up       255 pixels     2.6 ha


The accuracy is good but not perfect. Some **agriculture** pixels have high NDVI (green crops), so they were mistaken for **mangrove**. Mixing up two green classes is a very common problem in real classification too. Using more bands, or images from different seasons, usually helps.

## **Image Statistics**

In [7]:
print("Percentiles of NDVI (2, 50, 98):", np.percentile(ndvi, [2, 50, 98]).round(3))

counts, edges = np.histogram(ndvi, bins=[-1, 0, 0.2, 0.5, 1])
labels = ["Water (<0)", "Bare (0-0.2)", "Sparse (0.2-0.5)", "Dense (>0.5)"]
for label, c in zip(labels, counts):
    print(f"{label:<17}{c:>6} {'#' * (c // 60)}")

Percentiles of NDVI (2, 50, 98): [-0.795  0.571  0.92 ]
Water (<0)         1207 ####################
Bare (0-0.2)        201 ###
Sparse (0.2-0.5)    136 ##
Dense (>0.5)       3256 ######################################################


## **Handling NoData and Clouds with NaN**
Pixels with **no valid data** (clouds, image edges) are often marked with **NaN** (Not a Number). Normal functions like `mean()` give NaN if even one NaN is present. NumPy has special **nan-functions** that skip them.

In [8]:
ndvi_cloudy = ndvi.copy()
ndvi_cloudy[20:30, 40:60] = np.nan           # A cloud covers this area

print("mean()   :", ndvi_cloudy.mean())
print("nanmean():", np.nanmean(ndvi_cloudy).round(3))
print("Cloud pixels:", np.isnan(ndvi_cloudy).sum())

mean()   : nan
nanmean(): 0.354
Cloud pixels: 200


## **Safe Division**
If NIR + Red is 0 (for example, for NoData pixels stored as 0), the division gives a warning and NaN. We can handle it safely:

In [9]:
a = np.array([0.4, 0.0, 0.3])
b = np.array([0.1, 0.0, 0.2])
safe_ndvi = np.divide(a - b, a + b, out=np.zeros_like(a), where=(a + b) != 0)
print(safe_ndvi)

[0.6 0.  0.2]


In [10]:
# Quick question
# What does np.where(ndvi > 0.5, 1, 0).sum() give us?